### Building a RAG System with LangChain & ChromaDB

Introduction

Retrival-Augmented Generation (RAG) is a technique which combines the capabilites of LLMs with external knowledge. 
- LangChain : A framework for developing applications powered by Language models
- ChromaDB : An open source vector store for storing and retrieving embeddings 

In [32]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
# langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

# importing the ulities
import numpy as np
import matplotlib.pyplot as plt
from typing import List

### RAG Architecture Overview

#### What is RAG?

Retrieval-Augmented Generation (RAG) is an AI architecture that enhances Large Language Models (LLMs) by providing relevant external knowledge during inference. Instead of relying solely on the model's training data, RAG retrieves relevant information from a knowledge base and uses it to generate accurate, context-aware responses.

---

#### RAG Workflow

text Documents     │     ▼ Document Loaders     │     ▼ Text Preprocessing     │     ▼ Text Chunking     │     ▼ Embedding Model     │     ▼ Vector Database     │     ▼ Retriever     │ User Query     │     ▼ Query Embedding     │     ▼ Similarity Search     │     ▼ Relevant Chunks     │     ▼ Prompt Construction     │     ▼ Large Language Model     │     ▼ Generated Response 

---

#### Components

##### 1. Document Loaders

Document loaders ingest data from various sources such as:

- PDF files
- Text documents
- Word documents
- CSV files
- Web pages
- Databases

The loader converts raw data into LangChain Document objects.

---

##### 2. Text Preprocessing

Raw text is cleaned before processing:

- Remove unnecessary whitespace
- Normalize formatting
- Remove noise and special characters
- Standardize document structure

This improves retrieval quality.

---

##### 3. Text Chunking

Large documents are divided into smaller chunks.

Common strategies:

- Character-based chunking
- Recursive chunking
- Token-based chunking
- Semantic chunking

Benefits:

- Better retrieval accuracy
- Reduced context window usage
- Improved relevance matching

---

##### 4. Embedding Generation

Each chunk is converted into a numerical vector representation using an embedding model.

Examples:

- sentence-transformers/all-MiniLM-L6-v2
- BAAI/bge-small-en-v1.5
- OpenAI Embeddings

Embeddings capture semantic meaning rather than exact keywords.

---

##### 5. Vector Database

Embeddings are stored in a vector database.

Popular options:

- Chroma
- FAISS
- Pinecone
- Weaviate
- Milvus

The vector database enables efficient similarity search.

---

##### 6. Retrieval

When a user submits a query:

1. The query is converted into an embedding.
2. Similarity search is performed.
3. The most relevant chunks are retrieved.

This process ensures that only relevant information is passed to the LLM.

---

##### 7. Prompt Augmentation

Retrieved chunks are combined with the user's query.

Example:

Context: [Retrieved Documents]  Question: [User Query]

This augmented prompt provides the LLM with relevant knowledge.

---

##### 8. Response Generation

The LLM uses:

- User question
- Retrieved context
- Prompt instructions

to generate an accurate and context-aware response.

---

#### Benefits of RAG

- Reduces hallucinations
- Uses up-to-date information
- Enables domain-specific knowledge
- Improves response accuracy
- Works with private enterprise data
- Reduces dependency on model retraining

---

#### Technology Stack

##### Data Ingestion

- LangChain Document Loaders
- PyPDFLoader
- TextLoader

##### Text Processing

- RecursiveCharacterTextSplitter
- TokenTextSplitter

##### Embeddings

- HuggingFace Embeddings
- Sentence Transformers

##### Vector Store

- ChromaDB
- FAISS

##### LLM

- Groq
- Ollama
- OpenAI

##### Framework

- LangChain
- LangGraph

---

#### Future Enhancements

- Hybrid Search
- Multi-Query Retrieval
- Context Compression
- Reranking
- Multi-Agent RAG
- Multimodal RAG
- Conversational Memory
- Knowledge Graph Integration

---

#### Conclusion

RAG combines retrieval systems with Large Language Models to create intelligent applications capable of generating accurate, context-aware responses from external knowledge sources. It forms the foundation of modern AI assistants, document chatbots, enterprise search systems, and agentic AI workflows.

### 1. Sample Data

In [34]:
# create the sample data
sample_data = [
    """
    Python is a high-level, interpreted programming language known for its
    simplicity, readability, and versatility. It supports multiple programming
    paradigms including procedural, object-oriented, and functional programming.

    Python is widely used in web development, artificial intelligence,
    machine learning, data science, automation, cybersecurity, and scientific
    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.

    The Python ecosystem contains thousands of open-source libraries that
    accelerate development and make it one of the most popular programming
    languages in the world.
    """,

    """
    Retrieval-Augmented Generation (RAG) is an architecture that combines
    information retrieval with large language models. Instead of relying solely
    on the model's training data, RAG systems retrieve relevant information
    from an external knowledge base.

    The retrieved documents are provided to the language model as additional
    context before generating a response. This improves factual accuracy,
    reduces hallucinations, and enables the model to answer questions about
    proprietary or recently updated information.

    Modern RAG systems often use vector databases, embeddings, reranking,
    hybrid search, and contextual compression to improve retrieval quality.
    """,

    """
    LangChain is an open-source framework designed to simplify the development
    of applications powered by large language models. It provides abstractions
    for prompts, chains, agents, memory systems, document loaders, and
    retrieval pipelines.

    Developers can integrate multiple AI models, external APIs, databases,
    and tools into a unified workflow. LangChain is widely used for building
    chatbots, question-answering systems, AI assistants, and RAG applications.

    The framework works with providers such as OpenAI, Anthropic, Groq,
    Ollama, Hugging Face, and many others.
    """,

    """
    Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone, Weaviate, Milvus,
    Qdrant, and FAISS. These systems enable semantic search rather than
    traditional keyword matching.
    """,

    """
    Machine Learning is a branch of Artificial Intelligence focused on building
    systems that learn patterns from data. Instead of explicitly programming
    every rule, developers train models using datasets and optimization
    algorithms.

    Common categories include supervised learning, unsupervised learning,
    reinforcement learning, and self-supervised learning. Machine learning
    powers recommendation systems, fraud detection, computer vision, speech
    recognition, and predictive analytics.

    The rapid growth of machine learning has significantly contributed to the
    development of modern AI systems and large language models.
    """
]

In [35]:
# save the sample data to a text file
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_data):
    with open(os.path.join(temp_dir, f"doc_{i}.txt"), "w") as f:
        f.write(doc)

print(f"Sample data saved to: {temp_dir}")

Sample data saved to: /var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8


In [36]:
temp_dir

'/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8'

### 2. Document Loading

In [37]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# load the documents using the TextLoader
loader = DirectoryLoader(
    temp_dir, 
    glob="*.txt",
    show_progress=True,
    loader_kwargs={"encoding": "utf-8"},
    loader_cls= TextLoader
)
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"First document content:\n{documents[0].page_content[:500]}...")


100%|██████████| 5/5 [00:00<00:00, 6713.03it/s]

Number of documents loaded: 5
First document content:

    Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone, Weavi...


In [38]:
documents

[Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8/doc_3.txt'}, page_content='\n    Vector databases are specialized storage systems designed to manage and\n    search high-dimensional vector embeddings efficiently. They play a crucial\n    role in modern AI applications, particularly Retrieval-Augmented Generation.\n\n    When documents are converted into embeddings, they are stored inside a\n    vector database. User queries are also transformed into embeddings and\n    compared using similarity search algorithms.\n\n    Popular vector databases include ChromaDB, Pinecone, Weaviate, Milvus,\n    Qdrant, and FAISS. These systems enable semantic search rather than\n    traditional keyword matching.\n    '),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8/doc_2.txt'}, page_content='\n    LangChain is an open-source framework designed to simplify the development\n    of applications powered by large lang

### 3. Document Splitter

In [39]:
# intialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50, 
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")
print(f"First chunk content:\n{chunks[0].page_content[:500]}...")

Number of chunks created: 10
First chunk content:
Vector databases are specialized storage systems designed to manage and
    search high-dimensional vector embeddings efficiently. They play a crucial
    role in modern AI applications, particularly Retrieval-Augmented Generation.

    When documents are converted into embeddings, they are stored inside a
    vector database. User queries are also transformed into embeddings and
    compared using similarity search algorithms.

    Popular vector databases include ChromaDB, Pinecone,...


In [40]:
chunks

[Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8/doc_3.txt'}, page_content='Vector databases are specialized storage systems designed to manage and\n    search high-dimensional vector embeddings efficiently. They play a crucial\n    role in modern AI applications, particularly Retrieval-Augmented Generation.\n\n    When documents are converted into embeddings, they are stored inside a\n    vector database. User queries are also transformed into embeddings and\n    compared using similarity search algorithms.\n\n    Popular vector databases include ChromaDB, Pinecone,'),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqvb_t8/doc_3.txt'}, page_content='vector databases include ChromaDB, Pinecone, Weaviate, Milvus,\n    Qdrant, and FAISS. These systems enable semantic search rather than\n    traditional keyword matching.'),
 Document(metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmpigqv

### 4. Embedding Model

In [41]:
# Initialize the HuggingFaceEmbeddings with a specific model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8477.33it/s]


model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


#### Initialize the ChromaDB Vector Store to store the chunks in vector representation

In [42]:
# Initialize the Chroma vector database
persist_directory="./chroma_db"
vectorestore = Chroma.from_documents(
    collection_name="rag_documents",
    documents=chunks,
    embedding = embedding_model,
    persist_directory=persist_directory
)

print("Vector store created and documents indexed successfully!")
print(f"Collection name: {vectorestore._collection_name}")
print(f"Number of documents in the vector store: {vectorestore._collection.count()}")
print(f"Persist directory: {persist_directory}")

Vector store created and documents indexed successfully!
Collection name: rag_documents
Number of documents in the vector store: 100
Persist directory: ./chroma_db


#### Text Similarity Search

In [43]:
query = "What is Python used for?"

similar_docs = vectorestore.similarity_search(query, k=3)
similar_docs

[Document(id='269e776a-58d0-4df7-99e7-aecfea703e22', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including procedural, object-oriented, and functional programming.\n\n    Python is widely used in web development, artificial intelligence,\n    machine learning, data science, automation, cybersecurity, and scientific\n    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.\n\n    The Python ecosystem'),
 Document(id='d93daebd-3ef4-45d4-8ba8-28b31c59defb', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmprmcmwi97/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including p

#### Advanced Similarity with Score

In [44]:
vectorestore.similarity_search_with_score(query, k=3)

[(Document(id='269e776a-58d0-4df7-99e7-aecfea703e22', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmp0ll_i6mq/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\n    paradigms including procedural, object-oriented, and functional programming.\n\n    Python is widely used in web development, artificial intelligence,\n    machine learning, data science, automation, cybersecurity, and scientific\n    computing. Popular frameworks include Django, Flask, FastAPI, and Streamlit.\n\n    The Python ecosystem'),
  0.40170004963874817),
 (Document(id='d93daebd-3ef4-45d4-8ba8-28b31c59defb', metadata={'source': '/var/folders/zb/z5p3pbrx5xl_bdxy0pkbj91m0000gn/T/tmprmcmwi97/doc_0.txt'}, page_content='Python is a high-level, interpreted programming language known for its\n    simplicity, readability, and versatility. It supports multiple programming\

In [45]:
from langchain_groq import ChatGroq

# Initialize the ChatGroq client
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [46]:
# test llm
test_lm = llm.invoke("What is the capital of France?")
print(test_lm)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.010936696, 'completion_tokens_details': None, 'prompt_time': 0.003861701, 'prompt_tokens_details': None, 'queue_time': 0.162346378, 'total_time': 0.014798397}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e8ee6-98a7-76e0-a5b6-d1fed2d38c3d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50}


### Modern RAG Chain

In [47]:
# converting the vectore store into a retriever
retriever = vectorestore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3} # top 3 similar chunks
    )


In [48]:
# create a prompt template
from langchain_core.prompts import ChatPromptTemplate
system_template = """You are a helpful assistant that answers questions based on the provided context.
If you don't know the answer, say you don't know. Always use the provided context to answer the question.
Context: {context}
Question: {input}
Answer:"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", "{input}")
])

In [49]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="You are a helpful assistant that answers questions based on the provided context.\nIf you don't know the answer, say you don't know. Always use the provided context to answer the question.\nContext: {context}\nQuestion: {input}\nAnswer:"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

#### CREATING THE DOCUMENT CHAIN

In [50]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="You are a helpful assistant that answers questions based on the provided context.\nIf you don't know the answer, say you don't know. Always use the provided context to answer the question.\nContext: {context}\nQuestion: {input}\nAnswer:"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inpu

In [51]:
## Create the Final RAG Chain
from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x128ab4a50>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="You are a helpful assistant that answers questions based on the provided context.\nIf you don't know the answer, say you don't know. 

In [52]:
response = rag_chain.invoke({"input": "What is Retrieval-Augmented Generation (RAG)?"})
response['answer']

"Retrieval-Augmented Generation (RAG) is an architecture that combines information retrieval with large language models. Instead of relying solely on the model's training data, RAG systems retrieve relevant information from an external knowledge base."

### Create RAG Chain Alternative using LCEL (LangChain Expression Language)

In [53]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel


In [54]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions based on the provided context."),
    ("human", "Context: {context}\nQuestion: {input}\nAnswer:")
])

chat_prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that answers questions based on the provided context.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Context: {context}\nQuestion: {input}\nAnswer:'), additional_kwargs={})])

In [55]:
# Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [56]:
# Build the chain using LCEL

rag_chain_lcel = (
    {
        'context': retriever | format_docs,
        'input': RunnablePassthrough()

    }
    | chat_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x128ab4a50>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that answers questions based on the provided context.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Context: {context}\nQuestion: {input}\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': Fa

In [57]:
reponse_lcel = rag_chain_lcel.invoke("What is machine learning?")
reponse_lcel

'Machine Learning is a branch of Artificial Intelligence focused on building systems that learn patterns from data. Instead of explicitly programming every rule, developers train models using datasets and optimization algorithms.'

In [58]:
docs = retriever.invoke("What is machine learning?")
for doc in docs:
    print(doc.page_content[:500], "\n---\n")

Machine Learning is a branch of Artificial Intelligence focused on building
    systems that learn patterns from data. Instead of explicitly programming
    every rule, developers train models using datasets and optimization
    algorithms.

    Common categories include supervised learning, unsupervised learning,
    reinforcement learning, and self-supervised learning. Machine learning
    powers recommendation systems, fraud detection, computer vision, speech
    recognition, and 
---

Machine Learning is a branch of Artificial Intelligence focused on building
    systems that learn patterns from data. Instead of explicitly programming
    every rule, developers train models using datasets and optimization
    algorithms.

    Common categories include supervised learning, unsupervised learning,
    reinforcement learning, and self-supervised learning. Machine learning
    powers recommendation systems, fraud detection, computer vision, speech
    recognition, and 
---

Machine Lear

In [59]:
# query function
def query_rag_lcel(query: str) -> str:
    return rag_chain_lcel.invoke(query)

print(query_rag_lcel("What is machine learning?"))

Machine Learning is a branch of Artificial Intelligence focused on building systems that learn patterns from data. Instead of explicitly programming every rule, developers train models using datasets and optimization algorithms.


### ADD new Document in the Existing Vector Store

In [61]:
vectorestore

In [63]:
# new document 
new_document = """Artificial Intelligence (AI) is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes problem-solving, learning, reasoning, perception, and natural language understanding. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task that a human can do. AI technologies are widely used in various applications such as virtual assistants, recommendation systems, autonomous vehicles, and healthcare diagnostics. The field continues to evolve rapidly, driven by advancements in machine learning, deep learning, and data availability."""

In [65]:
new_doc = Document(page_content=new_document, metadata={"source": "new_doc.txt", "topic": "Artificial Intelligence"})
new_doc

Document(metadata={'source': 'new_doc.txt', 'topic': 'Artificial Intelligence'}, page_content='Artificial Intelligence (AI) is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes problem-solving, learning, reasoning, perception, and natural language understanding. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task that a human can do. AI technologies are widely used in various applications such as virtual assistants, recommendation systems, autonomous vehicles, and healthcare diagnostics. The field continues to evolve rapidly, driven by advancements in machine learning, deep learning, and data availability.')

In [67]:
new_chunks =text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'new_doc.txt', 'topic': 'Artificial Intelligence'}, page_content='Artificial Intelligence (AI) is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes problem-solving, learning, reasoning, perception, and natural language understanding. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task that a human can do. AI technologies are widely used in various applications such as virtual assistants,'),
 Document(metadata={'source': 'new_doc.txt', 'topic': 'Artificial Intelligence'}, page_content='various applications such as virtual assistants, recommendation systems, autonomous vehicles, and healthcare diagnostics. The field continues to evolve rapidly, driven by advancements in machine learning, deep learning, and data availability.')]

In [68]:
# adding the new document to the vector store
vectorestore.add_documents(new_chunks)

['27fc47ff-8028-4268-83c9-352ce27ff6c6',
 '16c660ef-9607-4f31-892c-50506786003e']

In [69]:
query_rag_lcel("What is Artificial Intelligence?")

'Artificial Intelligence (AI) is a branch of computer science focused on creating systems capable of performing tasks that typically require human intelligence. This includes problem-solving, learning, reasoning, perception, and natural language understanding. AI can be categorized into narrow AI, which is designed for specific tasks, and general AI, which aims to perform any intellectual task that a human can do.'